# 02 — Data Understanding
## Guided examples with pandas

This notebook uses a small synthetic retail dataset created in memory. No external file is required. Run the cells from top to bottom.

**Goals:** inspect structure and types, understand row grain, profile missingness and uniqueness, validate values, and turn observations into questions for further analysis.


## 1. Import libraries and create a sample dataset
The dataset represents **one row per order**. A customer may therefore appear in multiple rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.DataFrame({
    "order_id": ["O1001", "O1002", "O1003", "O1004", "O1005", "O1006", "O1007", "O1008", "O1009", "O1010", "O1011", "O1012"],
    "customer_id": ["C01", "C02", "C01", "C03", "C04", "C05", "C02", "C06", "C07", "C08", "C09", "C10"],
    "order_date": ["2026-01-04", "2026-01-07", "2026-01-12", "2026-01-13", "2026-01-19", "2026-01-22", "2026-02-02", "2026-02-05", "2026-02-11", "2026-02-15", "2026-02-21", "2026-02-25"],
    "city": ["Indore", "Bhopal", "Indore", "indore", "Ujjain", "Bhopal", "Bhopal", "Indore", "Ujjain", None, "INDORE", "Bhopal"],
    "category": ["Electronics", "Home", "Books", "Electronics", "Books", "Home", "Electronics", "Books", "Home", "Books", "Electronics", "Home"],
    "quantity": [1, 2, 1, 1, 3, 2, 1, 2, 1, 2, 1, 2],
    "amount_inr": [24999, 1800, 499, 15999, 750, 2200, 12999, 600, 1750, np.nan, -500, 2600],
    "payment_method": ["UPI", "Card", "UPI", "Card", "Cash", "UPI", "Card", "UPI", "Cash", "Card", "UPI", "Unknown"],
    "returned": [False, False, False, False, False, True, False, False, False, False, True, False],
    "customer_age": [29, 35, 29, 42, 24, 31, 35, 27, 150, 38, 46, np.nan],
})
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders


## 2. Clarify the grain and inspect the schema
The row grain is one order. `customer_id` is expected to repeat when a customer places multiple orders; `order_id` should be unique.

In [ ]:
print("Rows, columns:", orders.shape)
print("Column names:", orders.columns.tolist())
orders.info()


In [ ]:
display(orders.head())
display(orders.sample(5, random_state=7))


## 3. Understand semantic types
Pandas storage dtypes are a starting point; interpret fields based on their meaning.

In [ ]:
type_guide = pd.DataFrame({
    "column": orders.columns,
    "storage_dtype": [str(orders[c].dtype) for c in orders.columns],
    "semantic_type": [
        "Identifier", "Identifier", "Date/time", "Nominal category",
        "Nominal category", "Discrete numeric", "Continuous numeric (INR)",
        "Nominal category", "Boolean", "Discrete numeric (years)"
    ],
})
type_guide


## 4. Descriptive summaries

In [ ]:
display(orders.describe())
display(orders.describe(include="all"))


## 5. Missingness and distinct values

In [ ]:
quality = pd.DataFrame({
    "dtype": orders.dtypes.astype(str),
    "missing_count": orders.isna().sum(),
    "missing_percent": (orders.isna().mean() * 100).round(2),
    "distinct_count_including_missing": orders.nunique(dropna=False),
})
quality.sort_values("missing_percent", ascending=False)


In [ ]:
for column in ["city", "category", "payment_method"]:
    print(f"\\n{column} frequency table")
    display(orders[column].value_counts(dropna=False))


## 6. Uniqueness and duplicates

In [ ]:
print("Fully duplicated rows:", int(orders.duplicated().sum()))
print("Duplicate order IDs:", int(orders["order_id"].duplicated().sum()))
print("Missing order IDs:", int(orders["order_id"].isna().sum()))
print("Unique customers:", int(orders["customer_id"].nunique()))
print("Repeated customer ID rows:", int(orders["customer_id"].duplicated().sum()))


## 7. Validate values without deleting anything
These are **flags for review**, not automatic proof of errors. Negative amounts may represent refunds, and the allowed age range should be confirmed with the data owner.

In [ ]:
checks = pd.DataFrame({
    "negative_amount": orders["amount_inr"].lt(0),
    "age_outside_0_120": orders["customer_age"].notna() & ~orders["customer_age"].between(0, 120),
    "missing_amount": orders["amount_inr"].isna(),
    "unknown_payment_method": orders["payment_method"].eq("Unknown"),
})
checks.sum().rename("flagged_records")


In [ ]:
display(orders.loc[checks.any(axis=1)].assign(**checks.loc[checks.any(axis=1)]))


## 8. Investigate inconsistent category labels
Normalization helps reveal variants. Confirm that labels refer to the same place before replacing them.

In [ ]:
orders["city_normalized_for_review"] = orders["city"].astype("string").str.strip().str.casefold()
display(orders[["city", "city_normalized_for_review"]].value_counts(dropna=False).rename("rows").reset_index())


## 9. Simple visual inspection

In [ ]:
ax = orders.groupby("category", dropna=False)["amount_inr"].sum(min_count=1).plot(kind="bar")
ax.set_title("Recorded amount by category (INR)")
ax.set_xlabel("Category")
ax.set_ylabel("Sum of recorded amounts (INR)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 10. Write evidence-based observations
The following is an example pattern. Fill in the blanks using outputs from your run.

- **Dataset grain:** one row represents ____________________.
- **Coverage:** the date range is ____________________.
- **Completeness:** the columns with missing values are ____________________.
- **Uniqueness:** the key field(s) and duplicate counts are ____________________.
- **Potential validity issues:** ____________________.
- **Possible category inconsistency:** ____________________.
- **Important caveat:** a negative amount could mean ____________________.
- **Questions for the data owner:** ____________________.
- **Recommendation:** ready for deeper EDA / clean first / clarify definitions, because ____________________.


## Key takeaways
Data understanding combines code, domain context, and careful documentation. Keep suspicious records visible until their meaning is confirmed. Do not infer causality from exploratory patterns.